这个文档是单云模型求双高斯速度

In [4]:
import sunpy
import sunpy.map
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d
from astropy.coordinates import SkyCoord
from matplotlib.patches import ConnectionPatch
import glob
from scipy.io import readsav
from scipy.ndimage import zoom
from scipy.special import wofz
from functools import partial

In [8]:
def voigt_single_cloud(lambda1, I_voigt, gamma_voigt, sigma_voigt,
                       I_cloud, mu_cloud, sigma_cloud, I_constant, S_l, I_0):

    z = (lambda1 + 1j * gamma_voigt) / (sigma_voigt * np.sqrt(2))
    voigt_component = np.real(wofz(z)) / (sigma_voigt * np.sqrt(2 * np.pi)) * I_voigt
    tau1 = I_cloud * np.exp(-0.5*((lambda1 - mu_cloud) / sigma_cloud) ** 2)
    cloud_component = (S_l - I_0) * (1 - np.exp(-tau1))
    return voigt_component + cloud_component + I_constant
# 双高斯 + Voigt + 常数
def voigt_double_cloud(lambda1, I_voigt, gamma_voigt, sigma_voigt,
                       I_cloud1, mu_cloud1, sigma_cloud1,
                       I_cloud2, mu_cloud2, sigma_cloud2, I_constant,S_l):
    z = (lambda1 + 1j * gamma_voigt) / (sigma_voigt * np.sqrt(2))
    voigt_component = np.real(wofz(z)) / (sigma_voigt * np.sqrt(2 * np.pi)) * I_voigt
    tau1 = I_cloud1 * np.exp(-((lambda1 - mu_cloud1) / sigma_cloud1) ** 2)
    cloud_component1 = S_l * np.exp(-tau1)
    tau2 = I_cloud2 * np.exp(-((lambda1 - mu_cloud2) / sigma_cloud2) ** 2)
    cloud_component2 = S_l * np.exp(-tau2)
    return voigt_component + cloud_component1 + cloud_component2 + I_constant


def voigt(lambda1, I_voigt, gamma_voigt, sigma_voigt):
    """
    Voigt 分量函数
    """
    z = (lambda1 + 1j * gamma_voigt) / (sigma_voigt * np.sqrt(2))
    voigt_component = np.real(wofz(z)) / (sigma_voigt * np.sqrt(2 * np.pi)) * I_voigt
    return voigt_component


def voigt_max_coefficient(gamma_voigt, sigma_voigt):
    """
    计算 Voigt 的最大系数，用于归一化
    """
    z_max = (1j * gamma_voigt) / (sigma_voigt * np.sqrt(2))
    return (sigma_voigt * np.sqrt(2 * np.pi)) / (np.real(wofz(z_max)))

In [5]:
dir=r'D:\Learning\PHD1st\magnetic_reconnecion\data\CHASE_Ha\RSM20240618T211711_0000_HA.fits'
rsm=fits.open(dir)
spectrum=rsm[1].data[:,795,1521]
lam = rsm[1].header['CRVAL3'] + np.arange(rsm[1].header['NAXIS3']) * rsm[1].header['CDELT3']

In [9]:
dir=r'D:\Learning\PHD1st\magnetic_reconnecion\data\CHASE_Ha\RSM20240618T211711_0000_HA.fits'
rsm=fits.open(dir)
spectrum=rsm[1].data[:,795,1521]
lam = rsm[1].header['CRVAL3'] + np.arange(rsm[1].header['NAXIS3']) * rsm[1].header['CDELT3']
# ========================
# 准备数据
# ========================
ha_left_index=50
ha_right_index=100
x_fit = lam[ha_left_index:ha_right_index + 1] - 6562.8 
y_fit = spectrum[ha_left_index:ha_right_index + 1]
S_l=[]
# Voigt 分量初值与范围
gamma_voigt_initial = 0.5
gamma_voigt_min, gamma_voigt_max = 0.0, 7

sigma_voigt_initial = 0.5
sigma_voigt_min, sigma_voigt_max = 0.0, 7

I_voigt_initial = 1.0 * np.max(y_fit) * voigt_max_coefficient(gamma_voigt_initial, sigma_voigt_initial)
I_voigt_min = 0.0
I_voigt_max = 5.0 * np.max(y_fit) * voigt_max_coefficient(gamma_voigt_initial, sigma_voigt_initial)

# 双高斯分量参数
I_cloud1_initial, I_cloud1_min, I_cloud1_max = 0.5, 0.01, 10
mu_cloud1_initial, mu_cloud1_min, mu_cloud1_max = -0.3, -1.0, -0.05
sigma_cloud1_initial, sigma_cloud1_min, sigma_cloud1_max = 0.1, 0.01, 0.5

I_cloud2_initial, I_cloud2_min, I_cloud2_max = 0.5, 0.01, 10
mu_cloud2_initial, mu_cloud2_min, mu_cloud2_max = 0.3, 0.05, 1.0
sigma_cloud2_initial, sigma_cloud2_min, sigma_cloud2_max = 0.1, 0.01, 0.5

# 常数项
I_constant_initial, I_constant_min, I_constant_max = 0.0, -0.01, 0.01
S_l_initial,S_l_min,S_l_max=0.003,0.001,1.0
# ========================
# 拟合调用（双高斯）
# ========================
popt, pcov = curve_fit(
    voigt_double_cloud, x_fit, y_fit,
    p0=(I_voigt_initial, gamma_voigt_initial, sigma_voigt_initial,
        I_cloud1_initial, mu_cloud1_initial, sigma_cloud1_initial,
        I_cloud2_initial, mu_cloud2_initial, sigma_cloud2_initial,
        I_constant_initial,S_l_initial),
    bounds=(
        [I_voigt_min, gamma_voigt_min, sigma_voigt_min,
         I_cloud1_min, mu_cloud1_min, sigma_cloud1_min,
         I_cloud2_min, mu_cloud2_min, sigma_cloud2_min,
         I_constant_min,S_l_min],
        [I_voigt_max, gamma_voigt_max, sigma_voigt_max,
         I_cloud1_max, mu_cloud1_max, sigma_cloud1_max,
         I_cloud2_max, mu_cloud2_max, sigma_cloud2_max,
         I_constant_max,S_l_max]
    )
)

# ========================
# 提取参数
# ========================
I_voigt_fit, gamma_voigt_fit, sigma_voigt_fit, \
I_cloud1_fit, mu_cloud1_fit, sigma_cloud1_fit, \
I_cloud2_fit, mu_cloud2_fit, sigma_cloud2_fit, \
I_constant_fit,S_l_fit = popt

# ========================
# 转换为速度（km/s）
# ========================
c = 3e5  # 光速 km/s
v1 = c * mu_cloud1_fit / 6562.8
v2 = c * mu_cloud2_fit / 6562.8

print(f"Velocity component 1 = {v1:.2f} km/s")
print(f"Velocity component 2 = {v2:.2f} km/s")


Velocity component 1 = -2.29 km/s
Velocity component 2 = 5.48 km/s
